The integrated variant effect table has combined scores but we need to filter out conflicting functional evidence and de-duplicate the variant sets.

In [ ]:
import pandas as pd

In [ ]:
bucket_mount_point = Path.home() / 'workspace/scalable-vus-validation'

full_table_file = bucket_mount_point / 'variant_tables/integrated_variant_effect_biobank_input_data_excalibr_assays_oddspath_models.tsv.gz'
filtered_table_file = bucket_mount_point / 'variant_tables/integrated_variant_effects_filtered.tsv.gz'

In [ ]:
full_df = pd.read_table(full_table_file, dtype=str)

In [ ]:
full_df

In [ ]:
deduped_df = full_df.assign(
    Combined_points=pd.to_numeric(full_df.Combined_points, downcast='integer'),
    Functional_points=pd.to_numeric(full_df.Functional_points, downcast='integer')
).groupby(
    ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']
).aggregate(
    conflicting=pd.NamedAgg(
        column='Functional_points',
        aggfunc=lambda x: (x.min() < 0) & (x.max() > 0)
    ),
    combined_points=pd.NamedAgg(
        column='Combined_points',
        aggfunc=lambda x: x.max() if x.max() > 0 else x.min()
    ),
    condensed_consequence=pd.NamedAgg(
        column='condensed_consequence',
        aggfunc=lambda x: x.to_list()[0]
    ),
    condensed_consequence_multiple=pd.NamedAgg(
        column='condensed_consequence',
        aggfunc=lambda x: x.drop_duplicates().to_list() if len(x.drop_duplicates()) > 1 else False
    )
)

In [ ]:
deduped_df

In [ ]:
deduped_df.describe()

In [ ]:
deduped_df.conflicting.value_counts()

In [ ]:
deduped_df.condensed_consequence_multiple.value_counts()

In [ ]:
deduped_df[deduped_df.condensed_consequence_multiple != False]

In [ ]:
deduped_df.to_csv(filtered_table_file, sep='\t')